In [1]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("/Users/suryaprkakash/Desktop/Machine-Learning-and-Deep-Learning/data/swiggy_rest_dataset.csv")

In [5]:
df.head()

,ID,Area,City,Restaurant,Price,Avg ratings,Total ratings,Food type,Address,Delivery time
0,211,Koramangala,Bangalore,Tandoor Hut,300.0,4.4,100,"Biryani,Chinese,North Indian,South Indian",5Th Block,59
1,221,Koramangala,Bangalore,Tunday Kababi,300.0,4.1,100,"Mughlai,Lucknowi",5Th Block,56
2,246,Jogupalya,Bangalore,Kim Lee,650.0,4.4,100,Chinese,Double Road,50
3,248,Indiranagar,Bangalore,New Punjabi Hotel,250.0,3.9,500,"North Indian,Punjabi,Tandoor,Chinese",80 Feet Road,57
4,249,Indiranagar,Bangalore,Nh8,350.0,4.0,50,"Rajasthani,Gujarati,North Indian,Snacks,Desser...",80 Feet Road,63


In [6]:
df.shape

(8680, 10)

In [7]:
df.isnull().sum()

ID               0
Area             0
City             0
Restaurant       0
Price            0
Avg ratings      0
Total ratings    0
Food type        0
Address          0
Delivery time    0
dtype: int64

In [9]:
df.describe()

,ID,Price,Avg ratings,Total ratings,Delivery time
count,8680.000000,8680.000000,8680.000000,8680.000000,8680.000000
mean,244812.071429,348.444470,3.655104,156.634793,53.967051
std,158671.617188,230.940074,0.647629,391.448014,14.292335
min,211.000000,0.000000,2.000000,20.000000,20.000000
25%,72664.000000,200.000000,2.900000,50.000000,44.000000
50%,283442.000000,300.000000,3.900000,80.000000,53.000000
75%,393425.250000,400.000000,4.200000,100.000000,64.000000
max,466928.000000,2500.000000,5.000000,10000.000000,109.000000


In [10]:
df = df.drop(columns=['ID'], inplace=True)

In [11]:
df.head()

,Area,City,Restaurant,Price,Avg ratings,Total ratings,Food type,Address,Delivery time
0,Koramangala,Bangalore,Tandoor Hut,300.0,4.4,100,"Biryani,Chinese,North Indian,South Indian",5Th Block,59
1,Koramangala,Bangalore,Tunday Kababi,300.0,4.1,100,"Mughlai,Lucknowi",5Th Block,56
2,Jogupalya,Bangalore,Kim Lee,650.0,4.4,100,Chinese,Double Road,50
3,Indiranagar,Bangalore,New Punjabi Hotel,250.0,3.9,500,"North Indian,Punjabi,Tandoor,Chinese",80 Feet Road,57
4,Indiranagar,Bangalore,Nh8,350.0,4.0,50,"Rajasthani,Gujarati,North Indian,Snacks,Desser...",80 Feet Road,63


In [27]:
#Creating a column price category
df['Price category'] = df['Price'].apply(
    lambda x: "Low" if x<200 else "Medium" if x<500 else "High"
)

In [28]:
df['Category'] = df['Avg ratings'].apply(
    lambda x: "Good" if x>=4 else "Average" if x>=2 else "Bad"
)

In [29]:
df.head()

,Area,City,Restaurant,Price,Avg ratings,Total ratings,Food type,Address,Delivery time,Price catagory,Category,Price category,Combined features
0,Koramangala,Bangalore,Tandoor Hut,300.0,4.4,100,"Biryani,Chinese,North Indian,South Indian",5Th Block,59,Medium,Good,Medium,"Biryani,Chinese,North Indian,South Indian Medi..."
1,Koramangala,Bangalore,Tunday Kababi,300.0,4.1,100,"Mughlai,Lucknowi",5Th Block,56,Medium,Good,Medium,"Mughlai,Lucknowi Medium Good"
2,Jogupalya,Bangalore,Kim Lee,650.0,4.4,100,Chinese,Double Road,50,High,Good,High,Chinese High Good
3,Indiranagar,Bangalore,New Punjabi Hotel,250.0,3.9,500,"North Indian,Punjabi,Tandoor,Chinese",80 Feet Road,57,Medium,Average,Medium,"North Indian,Punjabi,Tandoor,Chinese Medium Av..."
4,Indiranagar,Bangalore,Nh8,350.0,4.0,50,"Rajasthani,Gujarati,North Indian,Snacks,Desser...",80 Feet Road,63,Medium,Good,Medium,"Rajasthani,Gujarati,North Indian,Snacks,Desser..."


In [30]:
#Combine features

df['Combined features'] = (
    df['Food type'] + " " +
    df['Price category'] + " " +
    df['Category']
)

In [31]:
df.head()

,Area,City,Restaurant,Price,Avg ratings,Total ratings,Food type,Address,Delivery time,Price catagory,Category,Price category,Combined features
0,Koramangala,Bangalore,Tandoor Hut,300.0,4.4,100,"Biryani,Chinese,North Indian,South Indian",5Th Block,59,Medium,Good,Medium,"Biryani,Chinese,North Indian,South Indian Medi..."
1,Koramangala,Bangalore,Tunday Kababi,300.0,4.1,100,"Mughlai,Lucknowi",5Th Block,56,Medium,Good,Medium,"Mughlai,Lucknowi Medium Good"
2,Jogupalya,Bangalore,Kim Lee,650.0,4.4,100,Chinese,Double Road,50,High,Good,High,Chinese High Good
3,Indiranagar,Bangalore,New Punjabi Hotel,250.0,3.9,500,"North Indian,Punjabi,Tandoor,Chinese",80 Feet Road,57,Medium,Average,Medium,"North Indian,Punjabi,Tandoor,Chinese Medium Av..."
4,Indiranagar,Bangalore,Nh8,350.0,4.0,50,"Rajasthani,Gujarati,North Indian,Snacks,Desser...",80 Feet Road,63,Medium,Good,Medium,"Rajasthani,Gujarati,North Indian,Snacks,Desser..."


In [36]:
#Convert text into vectors using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

feature_matrix = tfidf.fit_transform(df['Combined features'])

In [37]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(feature_matrix)

In [38]:
def recommend_restaurant(name, df, similarity):
    idx = df[df['Restaurant'] == name].index[0]

    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1],reverse=True)

    top = scores[1:6]

    for i in top:
        print(df.iloc[i[0]]['Restaurant'])

In [39]:
recommend_restaurant("Tandoor Hut", df, similarity)

Salem Rr Restaurant
Anil Wine N Dine
Hotel Safari
Taj Mughalai
Hotel Parijatha - G.A Road
